## 1. Setup

In [1]:
import time
import zipfile
from pathlib import Path

from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import (StructType, StructField, StringType,
                               IntegerType, DoubleType)

# Walk up from the notebook until we find the folder containing data/raw
def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "data" / "raw").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate project root containing data/raw")

PROJECT   = find_project_root(Path.cwd())
RAW       = PROJECT / "data" / "raw"
PROCESSED = PROJECT / "data" / "processed"
GTFS_ZIP  = RAW / "gtfs_north_west.zip"
GTFS_DIR  = RAW / "gtfs_north_west"

PROCESSED.mkdir(parents=True, exist_ok=True)
print("Project root   :", PROJECT)
print("GTFS zip exists:", GTFS_ZIP.exists())

Project root   : E:\BODS-project
GTFS zip exists: True


## 2. Extract the archive

Spark cannot read CSV files from inside a zip, so the archive is expanded once.
The 90 MB download expands to roughly 550 MB — a 6:1 ratio that is itself an
argument for the columnar Parquet format used in §10.

In [2]:
GTFS_DIR.mkdir(parents=True, exist_ok=True)

if not (GTFS_DIR / "stop_times.txt").exists():
    t0 = time.time()
    with zipfile.ZipFile(GTFS_ZIP) as z:
        z.extractall(GTFS_DIR)
    print(f"Extracted in {time.time() - t0:.1f}s")
else:
    print("Already extracted.")

total_mb = 0
for f in sorted(GTFS_DIR.glob("*.txt")):
    mb = f.stat().st_size / 1e6
    total_mb += mb
    print(f"  {f.name:<22} {mb:9.2f} MB")
print(f"  {'TOTAL':<22} {total_mb:9.2f} MB")

Already extracted.
  agency.txt                  0.01 MB
  calendar.txt                0.01 MB
  calendar_dates.txt          0.28 MB
  feed_info.txt               0.00 MB
  routes.txt                  0.04 MB
  shapes.txt                133.01 MB
  stop_times.txt            399.63 MB
  stops.txt                   2.38 MB
  trips.txt                  12.74 MB
  TOTAL                     548.10 MB


## 3. SparkSession

`spark.sql.shuffle.partitions` defaults to 200, which is far too many for a
single machine — each partition becomes a task, and on a dataset this size the
scheduling overhead outweighs the parallelism. It is set to 8 here, comfortably
above the 4-partition minimum the brief requires while matching a typical
laptop core count.

Adaptive Query Execution is disabled so that the partition counts reported
below are the ones actually used. AQE would coalesce partitions at runtime and
make the evidence harder to interpret.

In [3]:
spark = (SparkSession.builder
         .appName("ST5011CEM_GTFS_Ingestion")
         .master("local[*]")
         .config("spark.driver.memory", "4g")
         .config("spark.sql.shuffle.partitions", "8")
         .config("spark.sql.adaptive.enabled", "false")
         .config("spark.sql.files.maxPartitionBytes", "64m")
         .getOrCreate())

spark.sparkContext.setLogLevel("WARN")

print("Spark version      :", spark.version)
print("Master             :", spark.sparkContext.master)
print("Default parallelism:", spark.sparkContext.defaultParallelism)
print("Shuffle partitions :", spark.conf.get("spark.sql.shuffle.partitions"))
print("\nSpark UI:", spark.sparkContext.uiWebUrl)

Spark version      : 3.5.8
Master             : local[*]
Default parallelism: 20
Shuffle partitions : 8

Spark UI: http://ujwal:4040


## 4. Load with explicit schemas

`inferSchema=True` makes Spark read the entire file once just to work out column
types, then read it again to load the data. On a 400 MB file that doubles the
I/O for no benefit, because the GTFS specification already fixes the types.

`stop_times` is the large table so it gets an explicit schema. The reference
tables are small enough that inference costs nothing.

In [4]:
stop_times_schema = StructType([
    StructField("trip_id",        StringType(),  True),
    StructField("arrival_time",   StringType(),  True),   # NOT a timestamp - see §5
    StructField("departure_time", StringType(),  True),
    StructField("stop_id",        StringType(),  True),
    StructField("stop_sequence",  IntegerType(), True),
    StructField("stop_headsign",  StringType(),  True),
    StructField("pickup_type",    IntegerType(), True),
    StructField("drop_off_type",  IntegerType(), True),
    StructField("timepoint",      IntegerType(), True),
])

def load(name, schema=None):
    path = (GTFS_DIR / f"{name}.txt").as_posix()
    reader = spark.read.option("header", True).option("mode", "PERMISSIVE")
    return reader.schema(schema).csv(path) if schema else \
           reader.option("inferSchema", True).csv(path)

stop_times = load("stop_times", stop_times_schema)
trips      = load("trips")
routes     = load("routes")
stops      = load("stops")
agency     = load("agency")
calendar   = load("calendar")

# Spark is lazy: nothing has been read yet, only the plan built.
print("stop_times columns:", stop_times.columns)
print("trips columns     :", trips.columns)
print("agency columns    :", agency.columns)

stop_times columns: ['trip_id', 'arrival_time', 'departure_time', 'stop_id', 'stop_sequence', 'stop_headsign', 'pickup_type', 'drop_off_type', 'timepoint']
trips columns     : ['route_id', 'service_id', 'trip_id', 'trip_headsign', 'direction_id', 'block_id', 'shape_id', 'wheelchair_accessible', 'vehicle_journey_code']
agency columns    : ['agency_id', 'agency_name', 'agency_url', 'agency_timezone', 'agency_lang', 'agency_phone', 'agency_noc']


### Record counts

This is the evidence for the brief's 100,000-record threshold.

In [5]:
tables = [("stop_times", stop_times), ("trips", trips), ("routes", routes),
          ("stops", stops), ("agency", agency), ("calendar", calendar)]

total = 0
print(f"{'Table':<14}{'Rows':>12}{'Partitions':>13}")
print("-" * 39)
for name, df in tables:
    n = df.count()
    total += n
    print(f"{name:<14}{n:>12,}{df.rdd.getNumPartitions():>13}")
print("-" * 39)
print(f"{'TOTAL':<14}{total:>12,}")
print(f"\n100,000-record threshold met: {total >= 100_000}")

Table                 Rows   Partitions
---------------------------------------
stop_times       4,665,277           20
trips              110,764            4
routes               1,469            1
stops               35,969            1
agency                  68            1
calendar               244            1
---------------------------------------
TOTAL            4,813,791

100,000-record threshold met: True


## 5. Cleaning: the GTFS 24-hour problem

GTFS represents a journey that begins before midnight and continues afterwards
by letting the hour field exceed 23. A bus leaving at 23:50 and arriving at
00:21 the next morning is written `24:21:00`, not `00:21:00`.

Casting these to a timestamp produces **null**, silently discarding every
late-night service — exactly the trips most likely to run late. The count below
shows how many rows this affects.

The fix is to convert to seconds-since-service-start, keep a `next_day` flag,
and derive a clock hour separately.

In [6]:
affected = stop_times.filter(F.col("arrival_time") >= "24:00:00").count()
total_st = stop_times.count()
print(f"Rows with hour >= 24: {affected:,} ({100*affected/total_st:.2f}% of stop_times)")
print("These would become NULL under a naive timestamp cast.\n")

st = (stop_times
      .withColumn("arr_h", F.split("arrival_time", ":").getItem(0).cast("int"))
      .withColumn("arr_m", F.split("arrival_time", ":").getItem(1).cast("int"))
      .withColumn("arr_s", F.split("arrival_time", ":").getItem(2).cast("int")))

st = (st
      .withColumn("arrival_sec",  F.col("arr_h") * 3600 + F.col("arr_m") * 60 + F.col("arr_s"))
      .withColumn("next_day",     (F.col("arr_h") >= 24).cast("int"))
      .withColumn("arrival_hour", F.col("arr_h") % 24)
      .drop("arr_h", "arr_m", "arr_s"))

st = st.filter(F.col("arrival_sec").isNotNull())

print("Sample of after-midnight rows, correctly handled:")
st.filter(F.col("next_day") == 1) \
  .select("trip_id", "arrival_time", "arrival_sec", "arrival_hour", "next_day") \
  .show(5, truncate=False)

Rows with hour >= 24: 47,585 (1.02% of stop_times)
These would become NULL under a naive timestamp cast.

Sample of after-midnight rows, correctly handled:
+------------------------------------------+------------+-----------+------------+--------+
|trip_id                                   |arrival_time|arrival_sec|arrival_hour|next_day|
+------------------------------------------+------------+-----------+------------+--------+
|VJ003d18a06a781ad6a6f2d45e764c32ad516ef3f3|25:00:00    |90000      |1           |1       |
|VJ003d18a06a781ad6a6f2d45e764c32ad516ef3f3|25:01:00    |90060      |1           |1       |
|VJ003d18a06a781ad6a6f2d45e764c32ad516ef3f3|25:03:00    |90180      |1           |1       |
|VJ003d18a06a781ad6a6f2d45e764c32ad516ef3f3|25:04:00    |90240      |1           |1       |
|VJ003d18a06a781ad6a6f2d45e764c32ad516ef3f3|25:04:00    |90240      |1           |1       |
+------------------------------------------+------------+-----------+------------+--------+
only showing top

## 6. Repartitioning and caching

`stop_times` is read from a single large CSV, so Spark's file-split logic drives
the initial partition count. Repartitioning by `trip_id` puts all stops of a
journey together, which matters in notebook 02 where consecutive stops on a trip
are compared.

The timing below is the caching evidence for the report: the first `count()`
reads from disk, the second reads from memory.

In [7]:
print("Partitions before repartition:", st.rdd.getNumPartitions())

st = st.repartition(8, "trip_id")
print("Partitions after repartition :", st.rdd.getNumPartitions())

st.cache()

t0 = time.time(); n = st.count(); cold = time.time() - t0
t0 = time.time(); _ = st.count(); warm = time.time() - t0

print(f"\nRows: {n:,}")
print(f"count() cold (disk)  : {cold:6.2f}s")
print(f"count() warm (cached): {warm:6.2f}s")
print(f"Speed-up             : {cold/warm:6.1f}x")

Partitions before repartition: 20
Partitions after repartition : 8

Rows: 4,665,277
count() cold (disk)  :  14.60s
count() warm (cached):   0.11s
Speed-up             :  127.4x


## 7. Broadcast joins

The reference tables are tiny — a few thousand rows at most — while `stop_times`
has millions. A standard join would shuffle the large table across partitions.
Broadcasting sends a copy of each small table to every executor instead, so the
large table never moves.

`F.broadcast()` states the intent explicitly rather than relying on Spark's
automatic threshold, which makes the optimisation visible in the query plan and
in the report.

In [8]:
scheduled = (st
    .join(F.broadcast(trips.select("trip_id", "route_id", "service_id", "direction_id")), "trip_id")
    .join(F.broadcast(routes.select("route_id", "agency_id", "route_short_name")), "route_id")
    .join(F.broadcast(agency.select("agency_id", "agency_name")), "agency_id")
    .join(F.broadcast(stops.select("stop_id", "stop_name", "stop_lat", "stop_lon")), "stop_id"))

print("Joined rows:", f"{scheduled.count():,}")
print("Columns    :", len(scheduled.columns))

scheduled.select("agency_name", "route_short_name", "stop_name",
                 "stop_sequence", "arrival_time", "arrival_hour").show(5, truncate=False)

Joined rows: 4,665,277
Columns    : 21
+-------------+----------------+---------------+-------------+------------+------------+
|agency_name  |route_short_name|stop_name      |stop_sequence|arrival_time|arrival_hour|
+-------------+----------------+---------------+-------------+------------+------------+
|First Halifax|591             |Bus Station    |0            |22:29:00    |22          |
|First Halifax|591             |Culvert        |1            |22:31:00    |22          |
|First Halifax|591             |Turf Moor      |3            |22:32:00    |22          |
|First Halifax|591             |Harry Potts Way|2            |22:32:00    |22          |
|First Halifax|591             |Ridge Avenue   |5            |22:33:00    |22          |
+-------------+----------------+---------------+-------------+------------+------------+
only showing top 5 rows



In [9]:
# Confirm the broadcast strategy was actually chosen.
# Look for "BroadcastHashJoin" rather than "SortMergeJoin".
scheduled.explain(mode="simple")

== Physical Plan ==
*(5) Project [stop_id#3, agency_id#71, route_id#35, trip_id#0, arrival_time#1, departure_time#2, stop_sequence#4, stop_headsign#5, pickup_type#6, drop_off_type#7, timepoint#8, arrival_sec#377, next_day#391, arrival_hour#406, service_id#36, direction_id#39, route_short_name#72, agency_name#133, stop_name#99, stop_lat#100, stop_lon#101]
+- *(5) BroadcastHashJoin [stop_id#3], [stop_id#97], Inner, BuildRight, false
   :- *(5) Project [agency_id#71, route_id#35, trip_id#0, arrival_time#1, departure_time#2, stop_id#3, stop_sequence#4, stop_headsign#5, pickup_type#6, drop_off_type#7, timepoint#8, arrival_sec#377, next_day#391, arrival_hour#406, service_id#36, direction_id#39, route_short_name#72, agency_name#133]
   :  +- *(5) BroadcastHashJoin [agency_id#71], [agency_id#132], Inner, BuildRight, false
   :     :- *(5) Project [route_id#35, trip_id#0, arrival_time#1, departure_time#2, stop_id#3, stop_sequence#4, stop_headsign#5, pickup_type#6, drop_off_type#7, timepoint#8, 

## 8. PySpark SQL

The same DataFrame exposed as a SQL view, satisfying the brief's requirement to
use Spark SQL for complex queries and aggregations.

In [10]:
scheduled.createOrReplaceTempView("scheduled_stops")

print("Scheduled service volume by operator:")
spark.sql("""
    SELECT agency_name                    AS operator,
           COUNT(*)                       AS scheduled_stops,
           COUNT(DISTINCT trip_id)        AS trips,
           COUNT(DISTINCT route_id)       AS routes,
           COUNT(DISTINCT stop_id)        AS stops_served
    FROM scheduled_stops
    GROUP BY agency_name
    ORDER BY scheduled_stops DESC
""").show(20, truncate=False)

Scheduled service volume by operator:
+------------------------------------------+---------------+-----+------+------------+
|operator                                  |scheduled_stops|trips|routes|stops_served|
+------------------------------------------+---------------+-----+------+------------+
|Bee Network                               |1934677        |40267|581   |12534       |
|Arriva North West                         |754903         |15175|130   |5244        |
|Stagecoach Cumbria and North Lancashire   |472772         |12299|146   |5469        |
|Blackpool Transport                       |272941         |5066 |20    |1307        |
|Stagecoach Merseyside and South Lancashire|267335         |6400 |87    |3230        |
|Metrolink                                 |182711         |12792|24    |192         |
|The Blackburn Bus Company                 |121321         |1910 |16    |1150        |
|The Burnley Bus Company                   |90840          |1959 |22    |1221        |
|Warr

In [11]:
print("Scheduled departures by hour of day (peak identification):")
spark.sql("""
    SELECT arrival_hour AS hour,
           COUNT(*)     AS scheduled_stops
    FROM scheduled_stops
    GROUP BY arrival_hour
    ORDER BY hour
""").show(24)

Scheduled departures by hour of day (peak identification):
+----+---------------+
|hour|scheduled_stops|
+----+---------------+
|   0|          34150|
|   1|           7251|
|   2|           4720|
|   3|           4197|
|   4|           7863|
|   5|          47411|
|   6|         129897|
|   7|         220197|
|   8|         263666|
|   9|         300765|
|  10|         318926|
|  11|         322892|
|  12|         323620|
|  13|         325042|
|  14|         324499|
|  15|         328217|
|  16|         320580|
|  17|         311310|
|  18|         282131|
|  19|         218336|
|  20|         172671|
|  21|         151428|
|  22|         138919|
|  23|         106589|
+----+---------------+



## 9. Data profiling and quality assessment

Null counts, cardinality, and a coordinate sanity check. The bounding box test
matters because a stop with a bad coordinate would corrupt the haversine
distance matching in notebook 02.

In [12]:
check_cols = ["trip_id", "arrival_time", "arrival_sec", "stop_id",
              "stop_lat", "stop_lon", "agency_name", "route_short_name"]

print("Null counts:")
scheduled.select([
    F.sum(F.col(c).isNull().cast("int")).alias(c) for c in check_cols
]).show(truncate=False)

print("Cardinality:")
scheduled.select([
    F.countDistinct(c).alias(c) for c in
    ["trip_id", "route_id", "stop_id", "agency_id", "service_id"]
]).show(truncate=False)

Null counts:
+-------+------------+-----------+-------+--------+--------+-----------+----------------+
|trip_id|arrival_time|arrival_sec|stop_id|stop_lat|stop_lon|agency_name|route_short_name|
+-------+------------+-----------+-------+--------+--------+-----------+----------------+
|0      |0           |0          |0      |0       |0       |0          |0               |
+-------+------------+-----------+-------+--------+--------+-----------+----------------+

Cardinality:
+-------+--------+-------+---------+----------+
|trip_id|route_id|stop_id|agency_id|service_id|
+-------+--------+-------+---------+----------+
|110764 |1469    |35769  |68       |393       |
+-------+--------+-------+---------+----------+



In [13]:
print("Descriptive statistics:")
scheduled.select("arrival_sec", "stop_sequence", "stop_lat", "stop_lon") \
         .describe().show()

# Skewness and kurtosis, required by the brief's EDA section
print("Distribution shape of scheduled arrival times:")
scheduled.select(
    F.round(F.mean("arrival_sec"), 1).alias("mean_sec"),
    F.round(F.stddev("arrival_sec"), 1).alias("stddev_sec"),
    F.round(F.skewness("arrival_sec"), 3).alias("skewness"),
    F.round(F.kurtosis("arrival_sec"), 3).alias("kurtosis"),
).show()

Descriptive statistics:
+-------+-----------------+------------------+-------------------+------------------+
|summary|      arrival_sec|     stop_sequence|           stop_lat|          stop_lon|
+-------+-----------------+------------------+-------------------+------------------+
|  count|          4665277|           4665277|            4665277|           4665277|
|   mean|51680.69272778444|26.696251047901335|  53.57895127435871|-2.562900025664348|
| stddev|16836.23092166138|20.540439122268936|0.29886860492955364|0.3608303089344133|
|    min|                0|                 0|           51.06156|-4.251722199402544|
|    max|           120300|               128|          55.955473|          0.936863|
+-------+-----------------+------------------+-------------------+------------------+

Distribution shape of scheduled arrival times:
+--------+----------+--------+--------+
|mean_sec|stddev_sec|skewness|kurtosis|
+--------+----------+--------+--------+
| 51680.7|   16836.2|   0.162|  -0

In [14]:
# Outlier / integrity check: coordinates must fall inside the collection bbox
BBOX = (-2.75, 53.32, -1.90, 53.70)   # same box used by collect_avl.py

outside = scheduled.filter(
    (F.col("stop_lon") < BBOX[0]) | (F.col("stop_lon") > BBOX[2]) |
    (F.col("stop_lat") < BBOX[1]) | (F.col("stop_lat") > BBOX[3])
).count()

inside = scheduled.count() - outside
print(f"Stops inside collection bounding box : {inside:,}")
print(f"Stops outside                        : {outside:,}")
print("\nNote: the north_west GTFS region is wider than the Greater Manchester")
print("bounding box, so rows outside are expected and are filtered in notebook 02.")

Stops inside collection bounding box : 2,471,437
Stops outside                        : 2,193,840

Note: the north_west GTFS region is wider than the Greater Manchester
bounding box, so rows outside are expected and are filtered in notebook 02.


## 10. Persist to Parquet

Parquet is columnar and compressed, so later notebooks that read four columns
out of twenty-four do not pay for the other twenty. Partitioning by operator
lets Spark skip whole directories when a query filters on one.

This intermediate write is the checkpoint the brief asks about: notebooks 02–04
start from this file rather than re-parsing the CSVs.

In [15]:
OUT = (PROCESSED / "scheduled_stops").as_posix()

t0 = time.time()
(scheduled
 .write
 .mode("overwrite")
 .partitionBy("agency_id")
 .parquet(OUT))
elapsed = time.time() - t0

parquet_mb = sum(f.stat().st_size for f in (PROCESSED / "scheduled_stops").rglob("*.parquet")) / 1e6
csv_mb = (GTFS_DIR / "stop_times.txt").stat().st_size / 1e6

print(f"Written in {elapsed:.1f}s")
print(f"Source CSV : {csv_mb:8.1f} MB")
print(f"Parquet    : {parquet_mb:8.1f} MB")
print(f"Compression: {csv_mb/parquet_mb:8.1f}x")

Written in 10.7s
Source CSV :    399.6 MB
Parquet    :     67.6 MB
Compression:      5.9x


In [16]:
# Verify the round-trip
check = spark.read.parquet(OUT)
print("Rows read back :", f"{check.count():,}")
print("Partitions     :", check.rdd.getNumPartitions())
check.printSchema()

Rows read back : 4,665,277
Partitions     : 32
root
 |-- stop_id: string (nullable = true)
 |-- route_id: integer (nullable = true)
 |-- trip_id: string (nullable = true)
 |-- arrival_time: string (nullable = true)
 |-- departure_time: string (nullable = true)
 |-- stop_sequence: integer (nullable = true)
 |-- stop_headsign: string (nullable = true)
 |-- pickup_type: integer (nullable = true)
 |-- drop_off_type: integer (nullable = true)
 |-- timepoint: integer (nullable = true)
 |-- arrival_sec: integer (nullable = true)
 |-- next_day: integer (nullable = true)
 |-- arrival_hour: integer (nullable = true)
 |-- service_id: integer (nullable = true)
 |-- direction_id: integer (nullable = true)
 |-- route_short_name: string (nullable = true)
 |-- agency_name: string (nullable = true)
 |-- stop_name: string (nullable = true)
 |-- stop_lat: double (nullable = true)
 |-- stop_lon: double (nullable = true)
 |-- agency_id: string (nullable = true)



In [17]:
print("Spark UI still live at:", spark.sparkContext.uiWebUrl)
print("Take your screenshots now, then run the next cell.")

Spark UI still live at: http://ujwal:4040
Take your screenshots now, then run the next cell.


In [18]:
spark.stop()
print("Session stopped. Notebook 01 complete.")

Session stopped. Notebook 01 complete.
